# Laboratorio 3 — Detección de Anomalías en Tráfico de Red con Machine Learning

**Estudiante:** Cristian Cabana Sulca  
**Curso:** Seguridad Informática — Unidad IV  
**Dataset:** `lab3/network_traffic.csv` (10 000 registros, 30 días de tráfico de red)  
**Modelo:** Isolation Forest (no supervisado)

## Tarea 3.1 — Exploración y Preprocesamiento

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report
import joblib
import warnings
warnings.filterwarnings('ignore')

print('Librerias cargadas correctamente')

Librerias cargadas correctamente


In [2]:
# Carga del dataset
df = pd.read_csv('network_traffic.csv', parse_dates=['timestamp'])

print(f'Forma del dataset: {df.shape}')
print(f'\nColumnas: {list(df.columns)}')
print(f'\nDistribucion de etiquetas:')
print(df['label'].value_counts())
print(f'\nEstadisticas descriptivas:')
df.describe()

Forma del dataset: (10000, 10)

Columnas: ['timestamp', 'src_ip', 'dst_ip', 'dst_port', 'protocol', 'bytes_sent', 'bytes_recv', 'duration_sec', 'packets', 'label']

Distribucion de etiquetas:
label
normal     9500
anomaly     500
Name: count, dtype: int64

Estadisticas descriptivas:


,timestamp,dst_port,bytes_sent,bytes_recv,duration_sec,packets
count,10000,10000.000000,1.000000e+04,1.000000e+04,10000.000000,1.000000e+04
mean,2024-05-15 23:14:49.001700096,5272.963700,2.815289e+07,4.124360e+05,447.154662,1.605501e+04
min,2024-05-01 00:00:39,21.000000,1.500000e+01,0.000000e+00,0.000000,1.000000e+00
25%,2024-05-08 14:12:09.249999872,53.000000,5.544000e+03,1.328800e+04,8.507500,5.000000e+00
50%,2024-05-15 21:18:54.500000,3389.000000,2.233900e+04,5.529050e+04,21.435000,2.400000e+01
75%,2024-05-23 09:47:11,8080.000000,9.478175e+04,2.213258e+05,44.145000,1.100000e+02
max,2024-05-30 23:56:18,65460.000000,4.987050e+09,8.155783e+07,83028.150000,2.939448e+06
std,NaN,7348.395782,3.115671e+08,1.964278e+06,4530.488171,1.672859e+05


In [3]:
# Histogramas de bytes_sent y duration_sec
import os
os.makedirs('evidencias', exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['bytes_sent'], bins=50, color='steelblue', edgecolor='black', alpha=0.8)
axes[0].set_title('Distribucion de bytes_sent')
axes[0].set_xlabel('Bytes enviados')
axes[0].set_ylabel('Frecuencia')

axes[1].hist(df['duration_sec'], bins=50, color='darkorange', edgecolor='black', alpha=0.8)
axes[1].set_title('Distribucion de duration_sec')
axes[1].set_xlabel('Duracion (segundos)')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.savefig('evidencias/SCR-3.1_eda.png', dpi=120, bbox_inches='tight')
plt.show()
print('Histogramas guardados')

Histogramas guardados


In [4]:
# Verificacion y tratamiento de valores nulos
print('Valores nulos por columna:')
print(df.isnull().sum())

# Eliminar filas con nulos si existen
df = df.dropna()
print(f'\nFilas tras eliminar nulos: {len(df)}')

# Tratamiento de outliers extremos con clipeo en percentil 99
for col in ['bytes_sent', 'bytes_recv', 'duration_sec', 'packets']:
    p99 = df[col].quantile(0.99)
    outliers = (df[col] > p99).sum()
    df[col] = df[col].clip(upper=p99)
    print(f'{col}: {outliers} valores clipados al percentil 99 ({p99:.2f})')

Valores nulos por columna:
timestamp       0
src_ip          0
dst_ip          0
dst_port        0
protocol        0
bytes_sent      0
bytes_recv      0
duration_sec    0
packets         0
label           0
dtype: int64

Filas tras eliminar nulos: 10000
bytes_sent: 100 valores clipados al percentil 99 (27689027.11)
bytes_recv: 100 valores clipados al percentil 99 (6416516.15)
duration_sec: 100 valores clipados al percentil 99 (3589.83)
packets: 100 valores clipados al percentil 99 (100808.23)


In [5]:
# Feature Engineering: 2 nuevas variables derivadas

# ratio_bytes: relacion entre bytes enviados y recibidos (>1 = mas saliente que entrante)
df['ratio_bytes'] = df['bytes_sent'] / (df['bytes_recv'] + 1)

# bytes_por_segundo: tasa de transferencia de salida
df['bytes_por_segundo'] = df['bytes_sent'] / (df['duration_sec'] + 0.001)

print('Variables derivadas creadas:')
print(df[['ratio_bytes', 'bytes_por_segundo']].describe())

Variables derivadas creadas:
        ratio_bytes  bytes_por_segundo
count  1.000000e+04       1.000000e+04
mean   2.175087e+03       4.692298e+04
std    1.067595e+05       7.846853e+05
min    2.766446e-05       1.403965e-01
25%    6.381439e-02       2.619800e+02
50%    4.393979e-01       1.301482e+03
75%    3.031250e+00       6.871713e+03
max    9.151127e+06       5.551302e+07


In [6]:
# Normalizacion con StandardScaler
FEATURES = ['bytes_sent', 'bytes_recv', 'duration_sec', 'packets',
            'dst_port', 'ratio_bytes', 'bytes_por_segundo']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[FEATURES])

print(f'Features normalizadas: {FEATURES}')
print(f'Media post-escala (debe ser ~0): {X_scaled.mean(axis=0).round(4)}')
print(f'Std post-escala (debe ser ~1):  {X_scaled.std(axis=0).round(4)}')

Features normalizadas: ['bytes_sent', 'bytes_recv', 'duration_sec', 'packets', 'dst_port', 'ratio_bytes', 'bytes_por_segundo']
Media post-escala (debe ser ~0): [-0. -0.  0.  0. -0. -0.  0.]
Std post-escala (debe ser ~1):  [1. 1. 1. 1. 1. 1. 1.]


## Tarea 3.2 — Entrenamiento del Modelo Isolation Forest

In [7]:
# Entrenamiento del modelo (sin usar la columna label)
modelo = IsolationForest(
    contamination=0.05,
    n_estimators=100,
    random_state=42
)
modelo.fit(X_scaled)

# Predicciones: -1 = anomalia, 1 = normal
predicciones = modelo.predict(X_scaled)

df['prediccion'] = predicciones
print(f'Anomalias detectadas: {(predicciones == -1).sum()}')
print(f'Normal:               {(predicciones == 1).sum()}')

Anomalias detectadas: 500
Normal:               9500


In [8]:
# Metricas de evaluacion usando la columna label como ground truth
# Conversion: anomaly -> -1, normal -> 1
y_true = df['label'].map({'anomaly': -1, 'normal': 1})
y_pred = df['prediccion']

# Para sklearn las metricas usan 0/1: anomaly=1 (positivo), normal=0
y_true_bin = (y_true == -1).astype(int)
y_pred_bin = (y_pred == -1).astype(int)

precision = precision_score(y_true_bin, y_pred_bin)
recall    = recall_score(y_true_bin, y_pred_bin)
f1        = f1_score(y_true_bin, y_pred_bin)

print('=== Metricas de Evaluacion ===')
print(f'Precision : {precision:.4f}')
print(f'Recall    : {recall:.4f}')
print(f'F1-Score  : {f1:.4f}')
print()
print(classification_report(y_true_bin, y_pred_bin,
      target_names=['Normal', 'Anomalia']))

=== Metricas de Evaluacion ===
Precision : 0.6660
Recall    : 0.6660
F1-Score  : 0.6660

              precision    recall  f1-score   support

      Normal       0.98      0.98      0.98      9500
    Anomalia       0.67      0.67      0.67       500

    accuracy                           0.97     10000
   macro avg       0.82      0.82      0.82     10000
weighted avg       0.97      0.97      0.97     10000



In [9]:
# Matriz de confusion con seaborn
cm = confusion_matrix(y_true_bin, y_pred_bin)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Anomalia'],
            yticklabels=['Normal', 'Anomalia'], ax=ax)
ax.set_ylabel('Real')
ax.set_xlabel('Predicho')
ax.set_title('Matriz de Confusion — Isolation Forest')
plt.tight_layout()
plt.savefig('evidencias/SCR-3.2_metricas.png', dpi=120, bbox_inches='tight')
plt.show()
print('Matriz de confusion guardada')

Matriz de confusion guardada


## Tarea 3.3 — Interpretación y Umbral Dinámico

In [10]:
# Score de anomalia: valores mas negativos = mas anomalos
scores = modelo.decision_function(X_scaled)
df['anomaly_score'] = scores

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(range(len(scores)), sorted(scores), color='crimson', linewidth=0.5)
ax.axhline(y=0, color='black', linestyle='--', linewidth=1, label='Umbral por defecto (0)')
ax.set_xlabel('Registros (ordenados por score)')
ax.set_ylabel('Anomaly Score (decision_function)')
ax.set_title('Score de Anomalia para todos los registros')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Score minimo (mas anomalo): {scores.min():.4f}')
print(f'Score maximo (mas normal):  {scores.max():.4f}')

Score minimo (mas anomalo): -0.2643
Score maximo (mas normal):  0.2163


In [11]:
# Curva Umbral vs F1-Score para encontrar el umbral optimo
thresholds = np.linspace(scores.min(), scores.max(), 200)
f1_scores = []

for t in thresholds:
    y_pred_t = (scores < t).astype(int)
    f1_scores.append(f1_score(y_true_bin, y_pred_t, zero_division=0))

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(thresholds, f1_scores, color='steelblue', linewidth=2)
ax.axvline(x=best_threshold, color='red', linestyle='--',
           label=f'Umbral optimo: {best_threshold:.4f} (F1={best_f1:.4f})')
ax.set_xlabel('Umbral (threshold)')
ax.set_ylabel('F1-Score')
ax.set_title('Curva Umbral vs F1-Score')
ax.legend()
plt.tight_layout()
plt.savefig('evidencias/SCR-3.3_umbral_f1.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Umbral optimo: {best_threshold:.4f}')
print(f'F1-Score maximo: {best_f1:.4f}')

Umbral optimo: -0.0131
F1-Score maximo: 0.6736


In [12]:
# Top 10 registros mas anomalos
top10_anomalos = df.nsmallest(10, 'anomaly_score')
print('Top 10 registros mas anomalos:')
top10_anomalos[['timestamp','src_ip','dst_ip','dst_port','protocol',
                'bytes_sent','bytes_recv','duration_sec','packets',
                'label','anomaly_score']]

Top 10 registros mas anomalos:


,timestamp,src_ip,dst_ip,dst_port,protocol,bytes_sent,bytes_recv,duration_sec,packets,label,anomaly_score
9667,2024-05-14 18:52:35,10.0.1.254,14.125.240.42,80,TCP,6.978062e+06,155.0,1.39,84490.00,anomaly,-0.264287
3935,2024-05-02 05:32:06,10.0.1.206,76.196.246.10,80,ICMP,9.992469e+06,230.0,1.10,67206.00,anomaly,-0.262137
1652,2024-05-22 04:24:05,10.0.1.83,134.254.60.66,53,ICMP,8.746305e+06,11.0,7.41,76098.00,anomaly,-0.256254
5058,2024-05-05 10:16:01,10.0.3.101,61.47.234.82,53,UDP,7.440598e+06,10.0,5.95,57096.00,anomaly,-0.253142
2756,2024-05-21 00:36:46,10.0.0.136,91.240.118.172,443,TCP,7.245199e+06,54.0,11.98,95061.00,anomaly,-0.250414
2428,2024-05-18 01:17:02,10.0.3.25,185.220.101.45,8080,UDP,2.768903e+07,9692.0,3380.80,100808.23,anomaly,-0.248829
9726,2024-05-21 04:32:45,10.0.3.174,185.220.101.45,443,TCP,2.768903e+07,1016.0,2997.88,100808.23,anomaly,-0.247602
7663,2024-05-16 16:37:46,10.0.1.97,23.129.64.214,53,UDP,9.981411e+06,324.0,6.15,78889.00,anomaly,-0.247248
3373,2024-05-30 07:17:09,10.0.1.114,185.220.101.45,53,UDP,9.687834e+06,391.0,5.00,80654.00,anomaly,-0.246195
3275,2024-05-30 03:08:56,10.0.0.141,185.220.101.45,80,UDP,2.768903e+07,8554.0,3560.92,100808.23,anomaly,-0.246023


### Analisis del Top 10 de Registros mas Anomalos

Los registros con score de anomalia mas bajo (mas negativos) se destacan por las siguientes caracteristicas que los convierten en posibles amenazas reales:

1. **Volumen de datos inusualmente alto (`bytes_sent`):** Transferencias que superan ampliamente el promedio del dataset pueden indicar **exfiltracion de datos**. Un host interno enviando cientos de MB hacia una IP externa no conocida es una señal critica.

2. **`ratio_bytes` elevado (mucho mas enviado que recibido):** En comunicaciones normales cliente-servidor, el cliente recibe mas de lo que envia. Un ratio muy alto invierte esta logica y sugiere que el host es la fuente de la exfiltracion, no el receptor.

3. **`bytes_por_segundo` extremo:** Una tasa de transferencia muy alta en poco tiempo (`duration_sec` bajo con `bytes_sent` alto) puede indicar un **volcado masivo de informacion** o una herramienta automatizada de exfiltracion.

4. **Puertos de destino inusuales (`dst_port`):** Conexiones a puertos no estandar (fuera del rango 80, 443, 22) desde hosts internos hacia IPs externas pueden indicar **comunicacion con C2 (Command & Control)** o tunel de datos encubierto.

5. **Protocolo ICMP con datos:** El protocolo ICMP no deberia transportar grandes volumenes de datos. Si aparece con `bytes_sent` alto, puede indicar **ICMP tunneling**, una tecnica de exfiltracion encubierta.

Estos patrones combinados representan indicadores tipicos de compromiso (IOC) en un entorno empresarial y deben correlacionarse con alertas del SIEM para confirmar un incidente.

## Tarea 3.4 — Exportacion del Modelo

In [13]:
# Serializar modelo y scaler con joblib
import os
os.makedirs('evidencias', exist_ok=True)

joblib.dump(modelo, 'modelo_anomalias.pkl')
joblib.dump(scaler, 'scaler.pkl')

print('Modelo serializado en: lab3/modelo_anomalias.pkl')
print('Scaler serializado en: lab3/scaler.pkl')

Modelo serializado en: lab3/modelo_anomalias.pkl
Scaler serializado en: lab3/scaler.pkl
